In [1]:
!nvidia-smi

Tue May  5 04:36:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [43]:
%%writefile matrix_mul.cu
#include <cuda_runtime.h>
#include <bits/stdc++.h>

using namespace std;
using namespace std::chrono;

// CUDA kernel
__global__ void multiply(int* A, int* B, int* C, int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < K) {
        int sum = 0;
        for (int i = 0; i < N; i++) {
            sum += A[row * N + i] * B[i * K + col];
        }
        C[row * K + col] = sum;
    }
}

// Initialize matrix with random values
void initialize(int* matrix, int size) {
    for (int i = 0; i < size; i++) {
        matrix[i] = rand() % 10;
    }
}

// Print matrix (only for small sizes)
void print(int* matrix, int rows, int cols) {
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            cout << matrix[i * cols + j] << " ";
        }
        cout << "\n";
    }
    cout << "\n";
}

// Sequential multiplication
void sequentialMultiply(int* A, int* B, int* C, int M, int N, int K) {
    for (int i = 0; i < M; i++) {
        for (int j = 0; j < K; j++) {
            int sum = 0;
            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * K + j];
            }
            C[i * K + j] = sum;
        }
    }
}

int main() {
    // Matrix dimensions
    int M = 512, N = 512, K = 512;

    int* A = new int[M * N];
    int* B = new int[N * K];
    int* C = new int[M * K];

    initialize(A, M * N);
    initialize(B, N * K);

    // Device memory
    int *X, *Y, *Z;
    cudaMalloc(&X, M * N * sizeof(int));
    cudaMalloc(&Y, N * K * sizeof(int));
    cudaMalloc(&Z, M * K * sizeof(int));

    cudaMemcpy(X, A, M * N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, N * K * sizeof(int), cudaMemcpyHostToDevice);

    int THREADS = 16;

    dim3 threads(THREADS, THREADS);

    // ✅ Correct grid dimensions
    dim3 blocks(
        (K + THREADS - 1) / THREADS,
        (M + THREADS - 1) / THREADS
    );

    // ---------------- CPU ----------------
    auto start = high_resolution_clock::now();

    sequentialMultiply(A, B, C, M, N, K);

    auto stop = high_resolution_clock::now();
    auto cpu_time = duration_cast<microseconds>(stop - start);

    // ---------------- GPU ----------------
    start = high_resolution_clock::now();

    multiply<<<blocks, threads>>>(X, Y, Z, M, N, K);
    cudaDeviceSynchronize();  // ✅ important

    stop = high_resolution_clock::now();
    auto gpu_time = duration_cast<microseconds>(stop - start);

    // Copy result back
    cudaMemcpy(C, Z, M * K * sizeof(int), cudaMemcpyDeviceToHost);

    // CUDA error check
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        cout << "CUDA Error: " << cudaGetErrorString(err) << endl;
    }

    // Print (only if small)
    if (M <= 10) {
        cout << "Result Matrix:\n";
        print(C, M, K);
    }

    cout << "CPU Time: " << cpu_time.count() << " ms\n";
    cout << "GPU Time: " << gpu_time.count() << " ms\n";

    // Cleanup
    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z);

    return 0;
}

Overwriting matrix_mul.cu


In [44]:
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [45]:
!./matrix_mul

CPU Time: 661867 ms
GPU Time: 912 ms


In [46]:
%%writefile vector_addition.cu

#include <cuda_runtime.h>
#include <bits/stdc++.h>

using namespace std;
using namespace std::chrono;

// CUDA kernel
__global__ void add(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}

// Initialize with random values
void initialize(int* arr, int size) {
    for (int i = 0; i < size; i++) {
        arr[i] = rand() % 100;
    }
}

// Print first few elements (for verification)
void printSample(int* arr, int size, string name) {
    cout << name << " (first 10 elements): ";
    for (int i = 0; i < min(size, 10); i++) {
        cout << arr[i] << " ";
    }
    cout << "\n";
}

// Sequential addition
void sequentialAddition(int* A, int* B, int* C, int size) {
    for (int i = 0; i < size; i++) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    // 🔥 Large input
    int N = 1000000;  // 1 million elements

    int* A = new int[N];
    int* B = new int[N];
    int* C = new int[N];

    initialize(A, N);
    initialize(B, N);

    printSample(A, N, "Vector A");
    printSample(B, N, "Vector B");

    // Device memory
    int *X, *Y, *Z;
    cudaMalloc(&X, N * sizeof(int));
    cudaMalloc(&Y, N * sizeof(int));
    cudaMalloc(&Z, N * sizeof(int));

    cudaMemcpy(X, A, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(Y, B, N * sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // ---------------- CPU ----------------
    auto start = high_resolution_clock::now();

    sequentialAddition(A, B, C, N);

    auto stop = high_resolution_clock::now();
    auto cpu_time = duration_cast<microseconds>(stop - start);

    printSample(C, N, "C (CPU)");

    // ---------------- GPU ----------------
    start = high_resolution_clock::now();

    add<<<blocksPerGrid, threadsPerBlock>>>(X, Y, Z, N);
    cudaDeviceSynchronize();  // important

    stop = high_resolution_clock::now();
    auto gpu_time = duration_cast<microseconds>(stop - start);

    // Copy result back
    cudaMemcpy(C, Z, N * sizeof(int), cudaMemcpyDeviceToHost);

    // Error check
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        cout << "CUDA Error: " << cudaGetErrorString(err) << endl;
    }

    printSample(C, N, "C (GPU)");

    cout << "CPU Time: " << cpu_time.count() << " ms\n";
    cout << "GPU Time: " << gpu_time.count() << " ms\n";

    // Cleanup
    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(X);
    cudaFree(Y);
    cudaFree(Z);

    return 0;
}

Overwriting vector_addition.cu


In [47]:
!nvcc vector_addition.cu -o vector_addition

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [48]:
!./vector_addition

Vector A (first 10 elements): 83 86 77 15 93 35 86 92 49 21 
Vector B (first 10 elements): 89 63 84 93 81 55 6 93 61 50 
C (CPU) (first 10 elements): 172 149 161 108 174 90 92 185 110 71 
C (GPU) (first 10 elements): 172 149 161 108 174 90 92 185 110 71 
CPU Time: 4754 ms
GPU Time: 231 ms
